# Scoring: EmTract emotions and FinBERT polarity, step by step

This notebook walks through steps 06 and 07 of the pipeline on a handful of **made-up sentences** (no
real post, id or account is used): EmTract's stage 1 (text versions, EmTract's own cleaning with its
flags, tokens without cutting, the window plan for long texts), the model alone on the token ids, the
merge into one score row per text (`p`, `pc`, `pw`), and FinBERT's polarity on the original text. It
calls the same functions as `pipeline/06_emotion_emtract.py`, `pipeline/emtract_method.py`,
`pipeline/07_polarity_finbert.py` and `pipeline/polarity_method.py`, on the CPU.

**Environment.** A Jupyter kernel of the `emtract-ref` environment (`environment-emtract-ref.yml`; add
`ipykernel` to it to run the notebook interactively): EmTract's tokenizer needs tokenizers 0.10.3. The
model files must be in `data/raw/emtract_models/` and `data/raw/polarity_models/finbert/` (README.md).
Run it from `code/notebooks/`. The stored outputs below were produced on 2 Oct 2026 (Apple M2 Pro, CPU).

In [1]:
import hashlib, importlib.util, sys
from pathlib import Path
import numpy as np

CODE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE))
from pipeline import emtract_method as E
from pipeline import polarity_method as PM
spec = importlib.util.spec_from_file_location("step06", CODE / "pipeline" / "06_emotion_emtract.py")
M06 = importlib.util.module_from_spec(spec); spec.loader.exec_module(M06)
import torch, transformers, tokenizers
torch.set_num_threads(2)
print("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| tokenizers", tokenizers.__version__)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for b in iter(lambda: fh.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

def sums(folder):
    return {ln.split()[1].lstrip("*"): ln.split()[0] for ln in (folder / "SHA256SUMS").read_text().splitlines() if ln.strip()}

emtract_dir = E.EMTRACT_MODEL_DIR / E.MODEL_VARIANT
finbert_dir = PM.model_dir("finbert")
for name, folder, weights in (("EmTract stocktwits", emtract_dir, "pytorch_model.bin"), ("FinBERT", finbert_dir, "pytorch_model.bin")):
    got = sha256(folder / weights)
    print(f"{name:20s} {weights} sha256 {got[:16]}...  equals SHA256SUMS: {got == sums(folder)[weights]}")

python 3.9.15 | torch 1.11.0.post2 | transformers 4.10.3 | tokenizers 0.10.3
EmTract stocktwits   pytorch_model.bin sha256 e5d532f4cfa4590f...  equals SHA256SUMS: True
FinBERT              pytorch_model.bin sha256 e15a7b5738df7f17...  equals SHA256SUMS: True


## 1. EmTract stage 1: text versions, cleaning, flags, tokens

Every source row gets an `own` version; rows that quote another text also get `quoted` and `both`
(Reddit: the quoted lines and the original body; Twitter: the parent text and own + " " + parent).
EmTract's own `clean_text` cleans each version **once**; a NULL text is flagged `is_missing_text`, a text
that cleans to "" `is_empty_after_clean` (a cleaning error is flagged `is_clean_error` and also counts as
empty). The cleaned text is tokenized with tokenizers 0.10.3 **without cutting** and without special
tokens. Texts without tokens get no window and, later, no model call.

In [2]:
E.load_cleaning()
E.install_emoji_memo()                                   # exact memo of advertools' per-character lookup
tokens = E.Tokens(E.EMTRACT_MODEL_DIR / E.TOKENIZER_VARIANT)

reddit = {"text": ["> Is this the bottom?\nNo idea, but I am buying more ETH today",
                   "What a disgusting rug pull, the devs should be ashamed", ""],
          "own_text": ["No idea, but I am buying more ETH today", "What a disgusting rug pull, the devs should be ashamed", ""],
          "quoted_text": ["> Is this the bottom?", None, None],
          "has_quote_lines": [True, False, False], "is_dead": [False, False, True]}
stocktwits = {"message_body": ["$BTC.X just broke 20k, feeling great about this rally 🚀", None, "$DOGE"]}
twitter = {"text": ["I'm scared this crash will wipe out my savings", "Surprised nobody talks about this upgrade"],
           "parent_text": [None, "Ethereum merge happens next week"]}

for platform, cols in (("reddit", reddit), ("stocktwits", stocktwits), ("twitter", twitter)):
    out, counts = E.stage1_rows(platform, cols, tokens)
    print(f"--- {platform}: {counts['rows_in']} rows in, {counts['rows_dead']} dead, versions {counts['by_version']}")
    for i in range(len(out["version"])):
        flags = [f for f in ("is_missing_text", "is_empty_after_clean", "is_clean_error") if out[f][i]]
        print(f"  row {out['source_index'][i]} {out['version'][i]:6s} n_tokens={out['n_tokens'][i]:2d} "
              f"windows={out['window_starts'][i]} flags={flags} cleaned={out['cleaned_text'][i]!r}")

--- reddit: 3 rows in, 1 dead, versions {'own': 2, 'quoted': 1, 'both': 1}
  row 0 own    n_tokens=11 windows=[0] flags=[] cleaned='no idea , but i am buying more eth today'
  row 0 quoted n_tokens= 6 windows=[0] flags=[] cleaned='> is this the bottom ?'
  row 0 both   n_tokens=17 windows=[0] flags=[] cleaned='> is this the bottom ? no idea , but i am buying more eth today'
  row 1 own    n_tokens=12 windows=[0] flags=[] cleaned='what a disgusting rug pull , the devs should be ashamed'
--- stocktwits: 3 rows in, 0 dead, versions {'own': 3, 'quoted': 0, 'both': 0}
  row 0 own    n_tokens=17 windows=[0] flags=[] cleaned='<ticker> just broke <number> k , feeling great about this rally 🚀'
  row 1 own    n_tokens= 0 windows=[] flags=['is_missing_text'] cleaned=None
  row 2 own    n_tokens= 0 windows=[] flags=['is_empty_after_clean'] cleaned=''
--- twitter: 2 rows in, 0 dead, versions {'own': 2, 'quoted': 1, 'both': 1}
  row 0 own    n_tokens=10 windows=[0] flags=[] cleaned='i am scared this

## 2. Long texts: the window plan and the model inputs

The model sees at most 64 tokens: `[CLS]` + 62 wordpieces + `[SEP]`. Window 0 (`[CLS]` + the first 62
wordpieces + `[SEP]`) is exactly what EmTract's `predict()` gives the model, so it is the **primary**
input. A text of n > 62 wordpieces also gets k = ceil((n - 16) / 46) windows of 62 wordpieces that
overlap by at least 16 (a quarter window; Dai et al. 2022): the ideal starts are spread evenly from 0 to
n - 62, and each interior start moves to the nearest word start within 8 wordpieces if every overlap
stays at least 16.

In [3]:
long_text = " ".join(f"Bitcoin miners report record output number {i} while traders stay nervous about the next halving" for i in range(5))
out, _ = E.stage1_rows("stocktwits", {"message_body": [long_text]}, tokens)
n, starts, ids = out["n_tokens"][0], out["window_starts"][0], out["input_ids"][0]
print("n_tokens", n, "| k = ceil((n-16)/46) =", E.n_windows(n), "| ideal starts", E.ideal_starts(n), "| planned starts", starts)
E.check_window_plan(n, starts)                           # raises if the plan breaks a rule
inputs = E.model_inputs(ids, starts, tokens.cls_id, tokens.sep_id)
print("model inputs:", [len(x) for x in inputs], "tokens each; first starts with [CLS] =", inputs[0][0] == tokens.cls_id)
print("primary input == the tokenizer's own truncation to 64:", inputs[0] == tokens.encode_truncated([out["cleaned_text"][0]])[0])

n_tokens 100 | k = ceil((n-16)/46) = 2 | ideal starts [0, 38] | planned starts [0, 38]
model inputs: [64, 64] tokens each; first starts with [CLS] = True
primary input == the tokenizer's own truncation to 64: True


## 3. EmTract stage 2: the model alone on the token ids

`06_emotion_emtract.py pack` writes the model inputs of finished shards into batches that hold only
sequence numbers and token ids; `score` runs the stocktwits model on them in float32 (TF32 off on CUDA),
inputs sorted by length and padded to 16, 32, 48 or 64 tokens with an attention mask; `collect` checks
the raw scores and joins them back to shard, row and window. The full run scored 121,637,153 inputs in
26 batches on rented NVIDIA L4 GPUs; here the same model runs on the CPU, one input at a time.

In [4]:
model = transformers.DistilBertForSequenceClassification.from_pretrained(str(emtract_dir), num_labels=len(E.LABELS))
model = model.to(torch.float32).eval()

def emtract_probs(seqs):
    with torch.no_grad():
        return np.vstack([torch.softmax(model(input_ids=torch.tensor([s])).logits[0], -1).numpy() for s in seqs]).astype(np.float64)

P = emtract_probs(inputs)                                # one probability vector per window of the long text
print("labels:", E.LABELS)
for j, p in enumerate(P):
    print(f"window {j} (start {starts[j]:3d}):", np.round(p, 3), "top:", E.LABELS[int(p.argmax())])

labels: ('neutral', 'happy', 'sad', 'anger', 'disgust', 'surprise', 'fear')
window 0 (start   0): [0.021 0.032 0.005 0.003 0.002 0.006 0.933] top: fear
window 1 (start  38): [0.017 0.031 0.005 0.003 0.002 0.006 0.936] top: fear


## 4. Merge: one score row per text and version

* `p_<label>`: the primary score (window 0), what `predict()` would give;
* `pc_<label>`: the equal-weight mean of the k windows' probability vectors (the linear pool;
  Pappagari et al. 2019), set only when k >= 2;
* `pw_<label>`: the coverage-weighted mean (sensitivity check): token t is covered by c_t windows,
  window j gets w_j = sum over its tokens of 1/c_t (so sum_j w_j = n), set only when k >= 2.

Texts without tokens keep null scores (no model call). EmTract's `predict()` would give every empty
text one fixed score, that of `[CLS][SEP]`; it is computed once and recorded in every stats file.

In [5]:
w = M06.coverage_weights(n, tuple(starts))
print("coverage weights", np.round(w, 2), "| sum =", round(float(w.sum()), 6), "= n =", n)
p, pc, pw = P[0], P.mean(axis=0), (P * w[:, None]).sum(axis=0) / w.sum()
for name, v in (("p ", p), ("pc", pc), ("pw", pw)):
    print(name, np.round(v, 3), "top:", E.LABELS[int(v.argmax())])
empty = emtract_probs([[tokens.cls_id, tokens.sep_id]])[0]
print("empty-text score (recorded, never copied into rows):", np.round(empty, 3), "top:", E.LABELS[int(empty.argmax())])

coverage weights [50. 50.] | sum = 100.0 = n = 100
p  [0.021 0.032 0.005 0.003 0.002 0.006 0.933] top: fear
pc [0.019 0.032 0.005 0.003 0.002 0.006 0.934] top: fear
pw [0.019 0.032 0.005 0.003 0.002 0.006 0.934] top: fear
empty-text score (recorded, never copied into rows): [0.746 0.103 0.033 0.021 0.018 0.028 0.051] top: neutral


## 5. FinBERT polarity (step 07)

FinBERT (ProsusAI/finbert, pinned commit) scores the **original** text of the same rows and versions,
with no EmTract cleaning and no other preprocessing, using its own WordPiece tokenizer, uncut. The
primary input is `[CLS]` + the first 62 tokens + `[SEP]` (64, FinBERT's training length); longer texts
also get windows by the same rule (generalised to any window length). The polarity score is
P(positive) - P(negative), the score of FinBERT's own repository. Before any shard, `selfcheck` asserts
that the fast tokenizer gives exactly the stored ids of the slow reference tokenizer on 212 hand-written
texts. The full run scored 153,002,526 inputs on L4 GPUs under fp16 autocast and scored every 64th
input again in fp32: largest |dp| 0.0047, top label equal on at least 99.985% of the checked inputs.

In [6]:
ptok = PM.PolarityTokenizer(finbert_dir, 64)
spec = PM.MODELS["finbert"]
fin = transformers.BertForSequenceClassification.from_pretrained(str(finbert_dir)).to(torch.float32).eval()
labels = [fin.config.id2label[i].lower() for i in range(fin.config.num_labels)]
print("labels in the model's order:", labels, "| polarity = P(%s) - P(%s)" % (spec.pos_label, spec.neg_label))
texts = ["BTC just broke 20k, feeling great about this rally", "I'm scared this crash will wipe out my savings",
         "Ethereum merge happens next week", "   "]
ids, _ = ptok.encode(texts)
for t, x in zip(texts, ids):
    if not x:
        print(f"{t!r:55s} 0 tokens: no model call, NULL polarity")
        continue
    with torch.no_grad():
        pr = torch.softmax(fin(input_ids=torch.tensor([[ptok.prefix_id] + list(x[:62]) + [ptok.suffix_id]])).logits[0], -1).numpy()
    pol = pr[labels.index(spec.pos_label)] - pr[labels.index(spec.neg_label)]
    print(f"{t!r:55s} {len(x):2d} tokens  p={np.round(pr, 3)}  polarity={pol:+.3f}")

labels in the model's order: ['positive', 'negative', 'neutral'] | polarity = P(positive) - P(negative)
'BTC just broke 20k, feeling great about this rally'    12 tokens  p=[0.829 0.014 0.157]  polarity=+0.815
"I'm scared this crash will wipe out my savings"        11 tokens  p=[0.017 0.842 0.141]  polarity=-0.825
'Ethereum merge happens next week'                       6 tokens  p=[0.024 0.023 0.953]  polarity=+0.001
'   '                                                   0 tokens: no model call, NULL polarity


## 6. Running on the full data

The same functions run on the full corpora, one subcommand at a time (the headers of
`06_emotion_emtract.py` and `07_polarity_finbert.py` list the commands). `score` is plain PyTorch, so any
CUDA machine works, and the CPU or Apple MPS at lower speed.